# Start YOLO Pose Export / Fine-Tune Task

This notebook triggers the CHIMP training API task for the `YOLO Pose` plugin and polls task status until it finishes.


In [12]:
import json
import os
import time

import requests
from requests.adapters import HTTPAdapter
from requests.exceptions import ConnectionError, RequestException
from urllib3.util.retry import Retry

In [13]:
# --- Configure run options ---
TRAINING_SERVER_URL = os.environ.get("TRAINING_SERVER_URL", "http://localhost:5253")
PLUGIN_ROUTE_NAME = "YOLO+Pose"
RUN_URL = f"{TRAINING_SERVER_URL}/tasks/run/{PLUGIN_ROUTE_NAME}"

# Required by plugin
EXPERIMENT_NAME = "yolo_pose_demo"

DATASET_NAME = "hpe_one_image"

# Polling settings
POLL_INTERVAL_SECONDS = 2
POLL_TIMEOUT_SECONDS = 900

print("Training server:", TRAINING_SERVER_URL)
print("Run URL:", RUN_URL)
print("Experiment:", EXPERIMENT_NAME)
print("Dataset:", DATASET_NAME)

Training server: http://localhost:5253
Run URL: http://localhost:5253/tasks/run/YOLO+Pose
Experiment: yolo_pose_demo
Dataset: hpe_one_image


In [14]:
# Start task
form_data = {"experiment_name": EXPERIMENT_NAME}
if DATASET_NAME:
    form_data["dataset_name"] = DATASET_NAME

run_response = requests.post(RUN_URL, data=form_data, timeout=60)
print("Start status code:", run_response.status_code)

try:
    run_payload = run_response.json()
except Exception:
    run_payload = {"raw_text": run_response.text}

print(json.dumps(run_payload, indent=2))

if run_response.status_code != 200:
    raise RuntimeError("Failed to start YOLO task. See payload above.")

task_id = run_payload.get("task_id")
if not task_id:
    raise RuntimeError("No task_id returned by training API.")

POLL_URL = f"{TRAINING_SERVER_URL}/tasks/poll/{task_id}"
print("Task ID:", task_id)
print("Poll URL:", POLL_URL)

Start status code: 200
{
  "status": "task started successfully, use '/tasks/poll/aba0c4d9-47ec-44c1-9530-aeecc6c434d1' to poll for the current status",
  "task_id": "aba0c4d9-47ec-44c1-9530-aeecc6c434d1"
}
Task ID: aba0c4d9-47ec-44c1-9530-aeecc6c434d1
Poll URL: http://localhost:5253/tasks/poll/aba0c4d9-47ec-44c1-9530-aeecc6c434d1


In [15]:
# Poll until task reaches a terminal state
terminal_states = {"success", "successful", "failed", "failure", "error", "done", "completed"}
start_ts = time.time()
last_payload = None

session = requests.Session()
retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=0.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=("GET",),
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("http://", adapter)
session.mount("https://", adapter)

transient_failures = 0
max_transient_failures = 20

while True:
    try:
        poll_response = session.get(POLL_URL, timeout=60)
        poll_response.raise_for_status()
        try:
            poll_payload = poll_response.json()
        except ValueError:
            poll_payload = {"raw_text": poll_response.text}

        last_payload = poll_payload
        transient_failures = 0
        print(json.dumps(poll_payload, indent=2))

        if bool(poll_payload.get("ready", False)):
            break

        state = str(poll_payload.get("state", "")).lower()
        if state in terminal_states:
            break

    except ConnectionError as ex:
        transient_failures += 1
        print(
            f"Transient polling connection issue ({transient_failures}/{max_transient_failures}): {ex}"
        )
        if transient_failures >= max_transient_failures:
            raise TimeoutError(
                "Too many transient polling connection failures; aborting."
            ) from ex

    except RequestException as ex:
        transient_failures += 1
        print(
            f"Polling request error ({transient_failures}/{max_transient_failures}): {ex}"
        )
        if transient_failures >= max_transient_failures:
            raise RuntimeError(
                "Polling failed repeatedly due to HTTP/request errors."
            ) from ex

    if time.time() - start_ts > POLL_TIMEOUT_SECONDS:
        raise TimeoutError("Polling timed out before reaching terminal state.")

    time.sleep(POLL_INTERVAL_SECONDS)

print("Final poll payload:")
print(json.dumps(last_payload, indent=2))

{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null,
  "value": null
}
{
  "ready": false,
  "successful": null